<a href="https://colab.research.google.com/github/kaansoftware34/softito_calismalar_face2face/blob/main/finans_%C3%B6rne%C4%9Fi_ipynb_adl%C4%B1_not_120626.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install yfinance

In [5]:
import pandas as pd
import yfinance as yf

In [6]:
import pandas as pd
import yfinance as yf

# Sentetik veri yerine gerçek Apple (AAPL) hisse verilerini indirelim
df = yf.download('AAPL', start='2015-01-01', end='2024-01-01')

# Eksik satırları temizleyelim
df.dropna(inplace=True)

# İndirdiğimiz gerçek verinin ilk 5 satırını görelim
display(df.head())


/tmp/ipykernel_1908/704794776.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('AAPL', start='2015-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2015-01-02,24.192602,24.659504,23.754466,24.648440,212818400
2015-01-05,23.511063,24.042136,23.325188,23.962475,257142000
2015-01-06,23.513275,23.772173,23.152587,23.575233,263188400
2015-01-07,23.842981,23.942557,23.610636,23.721276,160423600
2015-01-08,24.759081,24.816614,24.053195,24.170475,237458000


In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [8]:
!pip install tensorflow
import tensorflow as tf
from tensorflow import keras

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2264 entries, 2015-01-02 to 2023-12-29
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, AAPL)   2264 non-null   float64
 1   (High, AAPL)    2264 non-null   float64
 2   (Low, AAPL)     2264 non-null   float64
 3   (Open, AAPL)    2264 non-null   float64
 4   (Volume, AAPL)  2264 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 106.1 KB


In [10]:
train_length = round(len(df)*0.7)
lg = len(df)
val_length = lg-train_length

print('Total observations:',lg)
print('Training set:', train_length)
print('Validation set:', val_length)


Total observations: 2264
Training set: 1585
Validation set: 679


In [11]:
# Kapanış (Close) fiyatlarını tahmin edeceğiz
# yfinance çıktısına uygun olarak veriyi seçelim (MultiIndex veya standart sütun olabilir)
if isinstance(df.columns, pd.MultiIndex):
    close_data = df['Close', 'AAPL']
else:
    close_data = df['Close']

# train_length değişkenini tanımlayalım
train_length = round(len(df)*0.7)

train_data = close_data[:train_length]
val_data = close_data[train_length:]


In [12]:
train=train_data.values.reshape(-1,1)
train

array([[ 24.19260216],
       [ 23.51106262],
       [ 23.51327515],
       ...,
       [130.55096436],
       [131.21266174],
       [129.52922058]])

In [13]:
scaler = MinMaxScaler()
scaled_trainset = scaler.fit_transform(train)

In [14]:
x_train = []
y_train = []
step = 50

for i in range(step, train_length):
    x_train.append(scaled_trainset[i-step:i,0])
    y_train.append(scaled_trainset[i,0])

In [15]:
X_train, y_train = np.array(x_train), np.array(y_train)

In [21]:
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
y_train = y_train.reshape(-1, 1)


In [17]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import SimpleRNN
from keras.layers import Dropout

In [20]:
from keras.layers import Input

model = Sequential()

# input_shape yerine ayrı bir Input katmanı ekliyoruz
model.add(Input(shape=(X_train.shape[1], 1)))

model.add(
    SimpleRNN(units = 50, return_sequences= True)
)

model.add(
    Dropout(0.2)
)

model.add(
    SimpleRNN(units = 50, return_sequences = True)
)

model.add(
    Dropout(0.2)
)

model.add(
    SimpleRNN(units = 50, return_sequences = True)
)

model.add(
    Dropout(0.2)
)

model.add(
    SimpleRNN(units = 50)
)

model.add(
    Dropout(0.2)
)

model.add(
    Dense(units = 1)
)


In [24]:
model.compile(optimizer = 'adam', loss = 'mean_squared_error', metrics = ['accuracy'])

In [25]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 50, 50)         │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 50, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 50, 50)         │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 50, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ (None, 50, 50)         │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 50, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,801 (69.54 KB)

 Trainable params: 17,801 (69.54 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
history = model.fit(X_train, y_train, epochs = 10, batch_size =16)

Epoch 1/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 6.5147e-04 - loss: 0.3095
Epoch 2/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 0.0013 - loss: 0.1380    
Epoch 3/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.0013 - loss: 0.0738    
Epoch 4/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.0013 - loss: 0.0488
Epoch 5/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - accuracy: 6.5147e-04 - loss: 0.0338
Epoch 6/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.0013 - loss: 0.0236
Epoch 7/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step - accuracy: 0.0013 - loss: 0.0207    
Epoch 8/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.0013 - loss: 0.0148
Epoch 9/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - accuracy: 0.0013 - loss: 0.0132
Epoch 10/10
96/96 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.0013 - loss: 0.0115    


In [27]:
y_pred = model.predict(X_train)
y_pred = scaler.inverse_transform(y_pred.reshape(1,-1))

48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step


In [28]:
X_train

array([[[0.03059629],
        [0.02484661],
        [0.02486527],
        ...,
        [0.05973908],
        [0.05812732],
        [0.06067617]],

       [[0.02484661],
        [0.02486527],
        [0.02764678],
        ...,
        [0.05812732],
        [0.06067617],
        [0.06459319]],

       [[0.02486527],
        [0.02764678],
        [0.03537528],
        ...,
        [0.06067617],
        [0.06459319],
        [0.06727321]],

       ...,

       [[0.92442265],
        [0.95270212],
        [0.94921373],
        ...,
        [0.93008578],
        [0.91038332],
        [0.93066016]],

       [[0.95270212],
        [0.94921373],
        [0.95044527],
        ...,
        [0.91038332],
        [0.93066016],
        [0.92786908]],

       [[0.94921373],
        [0.95044527],
        [0.94305641],
        ...,
        [0.93066016],
        [0.92786908],
        [0.93345137]]])

In [29]:
y_train = scaler.inverse_transform(y_train.reshape(1,-1))
y_train

array([[ 28.22242928,  28.54010582,  28.32461739, ..., 130.55096436,
        131.21266174, 129.52922058]])

In [30]:
y_train = scaler.inverse_transform(y_train.reshape(1,-1))
y_train

array([[ 3365.91566125,  3403.57149702,  3378.0285439 , ...,
        15495.4435799 , 15573.87797683, 15374.3310317 ]])

In [32]:
print(y_train.shape)
y_train = np.reshape(y_train, (-1, 1))


(1, 1535)


In [33]:
y_pred.shape

(1, 1535)

In [35]:
print(y_pred.shape)
y_pred = np.reshape(y_pred, (-1, 1))
y_pred

(1, 1535)


array([[ 28.599957],
       [ 27.947796],
       [ 28.216208],
       ...,
       [150.82367 ],
       [150.29034 ],
       [150.50601 ]], dtype=float32)

In [36]:
val = val_data.values.reshape(-1,1)
val

array([[129.90869141],
       [128.39067078],
       [130.70664978],
       [131.09588623],
       [130.77476501],
       [129.98657227],
       [129.88925171],
       [127.92359161],
       [128.97451782],
       [124.41069794],
       [124.6539917 ],
       [126.24986267],
       [126.92245483],
       [123.64728546],
       [122.7310257 ],
       [119.67028809],
       [121.81474304],
       [124.23210907],
       [123.08192444],
       [121.69778442],
       [121.54182434],
       [124.09567261],
       [122.26313019],
       [123.89097595],
       [123.69603729],
       [123.64728546],
       [122.11691284],
       [121.46383667],
       [121.14216614],
       [121.90245819],
       [120.42084503],
       [122.7115097 ],
       [122.72125244],
       [123.54004669],
       [123.92021942],
       [122.92593384],
       [124.13466644],
       [127.18560791],
       [126.36683655],
       [126.86392212],
       [128.46250916],
       [127.16611481],
       [128.95968628],
       [130

In [37]:
scaled_valset = scaler.fit_transform(val)

In [38]:
xval_train = []
yval_train = []
step = 50

for i in range(step, val_length):
    xval_train.append(scaled_valset[i-step:i,0])
    yval_train.append(scaled_valset[i,0])

In [39]:
X_val, y_val = np.array(xval_train), np.array(yval_train)

In [40]:
X_val = np.reshape(X_val, (X_val.shape[0],X_val.shape[1],1))  # reshape to 3D array
y_val = np.reshape(y_val, (-1,1))

In [41]:
y_pred_val = model.predict(X_val)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


In [42]:
y_val_is = scaler.inverse_transform(y_val)


In [51]:
import tensorflow as tf
import os
from tensorflow.keras.models import load_model

In [52]:
import os
from keras.models import load_model

# Klasör yoksa oluşturalım ki hata vermesin
os.makedirs('model', exist_ok=True)

model.save(os.path.join('model', 'SimpleRNN_Forecasting.keras'))
new_model = load_model(os.path.join('model', 'SimpleRNN_Forecasting.keras'))
